# 1. Tiền xử lý dữ liệu và chia time-bucket 4 giờ

Chuẩn hóa dữ liệu gốc, tạo Timestamp và gom nhóm theo chu kỳ 4 giờ.

In [28]:
import pandas as pd
import numpy as np
import warnings


# Pond_01 có Datetime hoàn chỉnh
url_pond01 = 'https://raw.githubusercontent.com/Ducanh-2003/aquaculture-risk-ontology/refs/heads/main/Data_Model_IoTMLCQ_2024.csv'
df_d1_raw = pd.read_csv(url_pond01)

# Pond_02 không có Datetime, dùng hour bucket
url_pond02 = 'https://raw.githubusercontent.com/Ducanh-2003/aquaculture-risk-ontology/refs/heads/main/Dataset_Model_IoTMLCQ_2024.csv'
df_d2_raw = pd.read_csv(url_pond02)

# Pond_03 có DateTime, chỉ có thông số môi trường
url_pond03 = 'https://raw.githubusercontent.com/Ducanh-2003/aquaculture-risk-ontology/refs/heads/main/Monteria_Aquaculture_Data_Hourly.csv'
df_d3_raw = pd.read_csv(url_pond03)

column_mapping = {
    'Disease Occurrence (Cases)': 'DiseaseOccurrence',
    'Temperature (°C)': 'Temp',
    'Temperature': 'Temp',
    'Dissolved Oxygen (mg/L)': 'DO',
    'Dissolved_Oxygen': 'DO',
    'pH': 'pH',
    'Turbidity (NTU)': 'Turbidity',
    'Turdidity': 'Turbidity',
    'Oxygenation Interventions': 'OxygenationInterventions'
}

d3_column_mapping = {
    'Temperature': 'Temp',
    'Dissolved_Oxygen': 'DO',
    'pH': 'pH',
    'Turdidity': 'Turbidity',
}

agg_funcs = {
    'Temp': 'mean',
    'pH': 'mean',
    'Turbidity': 'mean',
    'DO': 'min',
    'OxygenationInterventions': 'max',
    'DiseaseOccurrence': 'max'
}

d3_agg_funcs = {
    'Temp': 'mean',
    'pH': 'mean',
    'Turbidity': 'mean',
    'DO': 'min',
}
event_cols = ['DiseaseOccurrence', 'OxygenationInterventions']


def _finalize(df_grouped, pond_id, data_type):
    has_event_cols = [col for col in event_cols if col in df_grouped.columns]
    if has_event_cols:
        df_grouped[has_event_cols] = df_grouped[has_event_cols].fillna(0).astype(int)
    df_grouped = df_grouped.ffill().bfill()
    df_grouped['Pond_ID']   = pond_id
    df_grouped['Data_Type'] = data_type
    df_grouped['Observation_ID'] = (
        'Obs_' + pond_id + '_' + df_grouped['Timestamp'].dt.strftime('%Y%m%d_%H')
    )
    return df_grouped


def preprocess_d1(df_raw, pond_id, data_type):
    df = df_raw.copy()

    # Đọc Datetime trước khi rename để tránh lỗi do tên cột khác nhau giữa D1 và D2
    df['Datetime'] = pd.to_datetime(df['Datetime'], errors='coerce')
    df = df.dropna(subset=['Datetime'])

    df.rename(columns=column_mapping, inplace=True)

    df_grouped = (
        df.groupby(pd.Grouper(key='Datetime', freq='4h'))
          .agg(agg_funcs)
          .reset_index()
          .rename(columns={'Datetime': 'Timestamp'})
    )
    return _finalize(df_grouped, pond_id, data_type)


def preprocess_d2(df_raw, pond_id, data_type):
    df = df_raw.copy()

    df.rename(columns=column_mapping, inplace=True)

    df['Year'] = 2024
    df['hour_physical'] = df['hour'] * 4   

    df['Timestamp'] = pd.to_datetime(
        df[['Year', 'Month_Num', 'day', 'hour_physical']].rename(
            columns={'Month_Num': 'month', 'day': 'day', 'hour_physical': 'hour'}
        ),
        errors='coerce'
    )
    df = df.dropna(subset=['Timestamp'])

    df_grouped = (
        df.groupby(pd.Grouper(key='Timestamp', freq='4h'))
          .agg(agg_funcs)
          .reset_index()
    )
    return _finalize(df_grouped, pond_id, data_type)

def preprocess_d3(df_raw, pond_id, data_type):
    df = df_raw.copy()

    df['Datetime'] = pd.to_datetime(df['DateTime'], errors='coerce')
    df = df.dropna(subset=['Datetime'])

    df.rename(columns=d3_column_mapping, inplace=True)

    df_grouped = (
        df.groupby(pd.Grouper(key='Datetime', freq='4h'))
          .agg(d3_agg_funcs)
          .reset_index()
          .rename(columns={'Datetime': 'Timestamp'})
    )
    return _finalize(df_grouped, pond_id, data_type)

# Pond_01: tái tạo Timestamp từ hour bucket index
df_pond01 = preprocess_d1(
    df_d1_raw,
    pond_id='Pond_01',
    data_type='Real_Data',
)
print(f"\nPond_01 (Data_Model): dùng Datetime gốc | Rows: {len(df_pond01)}")

# Pond_02:  patch CorrectiveInterventions bằng mean của D1 
df_pond02 = preprocess_d2(
    df_d2_raw,
    pond_id='Pond_02',
    data_type='Real_Data',
)
print(f"Pond_02 (Dataset_Model): tái tạo Timestamp từ hour*4 | Rows: {len(df_pond02)}")

# Pond_03: chỉ có thông số môi trường, không có event, dùng Datetime gốc
df_pond03 = preprocess_d3(
    df_d3_raw,
    pond_id='Pond_03',
    data_type='Real_Data',
)
print(f"Pond_03 (Monteria): dùng Datetime gốc, không có event | Rows: {len(df_pond03)}")


df_real = df_pond01.copy()

display(df_pond01.head(3))
display(df_pond02.head(3))
display(df_pond03.head(3))
print(df_pond03.dtypes)


Pond_01 (Data_Model): dùng Datetime gốc | Rows: 1096
Pond_02 (Dataset_Model): tái tạo Timestamp từ hour*4 | Rows: 1080
Pond_03 (Monteria): dùng Datetime gốc, không có event | Rows: 1087


,Timestamp,Temp,pH,Turbidity,DO,OxygenationInterventions,DiseaseOccurrence,Pond_ID,Data_Type,Observation_ID
0,2024-01-01 00:00:00,27.47,7.98,3.3,6.34,1,2,Pond_01,Real_Data,Obs_Pond_01_20240101_00
1,2024-01-01 04:00:00,27.47,7.98,3.3,6.34,0,2,Pond_01,Real_Data,Obs_Pond_01_20240101_04
2,2024-01-01 08:00:00,27.47,7.98,3.3,6.34,1,2,Pond_01,Real_Data,Obs_Pond_01_20240101_08


,Timestamp,Temp,pH,Turbidity,DO,OxygenationInterventions,DiseaseOccurrence,Pond_ID,Data_Type,Observation_ID
0,2024-01-01 00:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20240101_00
1,2024-01-01 04:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20240101_04
2,2024-01-01 08:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20240101_08


,Timestamp,Temp,pH,Turbidity,DO,Pond_ID,Data_Type,Observation_ID
0,2024-01-01 00:00:00,27.679375,7.888962,3.609886,5.950694,Pond_03,Real_Data,Obs_Pond_03_20240101_00
1,2024-01-01 04:00:00,27.581754,7.925377,3.492159,5.678268,Pond_03,Real_Data,Obs_Pond_03_20240101_04
2,2024-01-01 08:00:00,27.171591,7.815301,3.353915,5.896264,Pond_03,Real_Data,Obs_Pond_03_20240101_08


Timestamp         datetime64[ns]
Temp                     float64
pH                       float64
Turbidity                float64
DO                       float64
Pond_ID                   object
Data_Type                 object
Observation_ID            object
dtype: object


# 2. Phân tích phân phối dữ liệu và percentiles

Tính các thống kê phân phối và percentiles để xác định ngưỡng baseline/pandemic.

In [29]:
# 2. Phân tích phân phối dữ liệu và percentiles
import pandas as pd

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
cols_to_analyze = [
    'DO', 'Temp', 'pH', 'Turbidity',
    'DiseaseOccurrence',
    'OxygenationInterventions'
]
distribution_stats = df_real[cols_to_analyze].describe(percentiles=percentiles).round(2)
display(distribution_stats.T)

P_baseline_disease_max       = df_real['DiseaseOccurrence'].quantile(0.75)   # = 1
P_pandemic_disease_threshold = df_real['DiseaseOccurrence'].quantile(0.90)   # = 2

print(f"DiseaseOccurrence sau group-by:")
print(f"  P75={P_baseline_disease_max:.1f} (baseline max) | P90={P_pandemic_disease_threshold:.1f} (outbreak threshold)")
print(f"  Tỉ lệ class=1 nếu dùng P90: {(df_real['DiseaseOccurrence'] >= P_pandemic_disease_threshold).mean():.1%}")

safe_DO_min = df_real['DO'].quantile(0.10)


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
DO,1096.0,6.93,0.63,6.34,6.34,6.34,6.34,6.39,6.68,7.72,7.89,7.89,7.89,7.89
Temp,1096.0,27.34,0.62,26.50,26.50,26.50,26.50,26.60,27.45,27.70,28.35,28.35,28.35,28.35
pH,1096.0,7.83,0.21,7.34,7.34,7.34,7.34,7.88,7.89,7.97,7.98,7.98,7.98,7.98
Turbidity,1096.0,3.31,0.43,2.79,2.79,2.79,2.79,2.79,3.30,3.71,3.98,3.98,3.98,3.98
DiseaseOccurrence,1096.0,1.15,0.36,1.00,1.00,1.00,1.00,1.00,1.00,1.00,2.00,2.00,2.00,2.00
OxygenationInterventions,1096.0,0.59,0.49,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00


DiseaseOccurrence sau group-by:
  P75=1.0 (baseline max) | P90=2.0 (outbreak threshold)
  Tỉ lệ class=1 nếu dùng P90: 15.0%


# 3. Tạo dữ liệu mô phỏng đa ao (Mock Data)

Sinh dữ liệu mô phỏng cho Pond_02 → Pond_05 dựa trên phân phối của Pond_01.

In [30]:
from datetime import datetime

df_raw = df_real.copy()

# Đồng bộ mốc thời gian với ngày hiện tại 
today_0h00 = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
max_raw_datetime = df_raw['Timestamp'].max().replace(hour=0, minute=0, second=0, microsecond=0)
time_shift = today_0h00 - max_raw_datetime
def shift_timestamps(df_in):
    df_out = df_in.copy()
    df_out['Timestamp'] = df_out['Timestamp'] + time_shift
    df_out['Observation_ID'] = 'Obs_' + df_out['Pond_ID'] + '_' + df_out['Timestamp'].dt.strftime('%Y%m%d_%H')
    return df_out
df_raw    = shift_timestamps(df_raw)
df_pond02 = shift_timestamps(df_pond02)
df_pond03 = shift_timestamps(df_pond03)


env_cols = ['Temp', 'DO', 'pH', 'Turbidity']
std = df_raw[env_cols].std()

mock_ponds = []
new_pond_ids = ['Pond_04', 'Pond_05']
np.random.seed(42)

for pond_id in new_pond_ids:
    df_mock = df_raw.copy()
    df_mock['Pond_ID'] = pond_id
    df_mock['Observation_ID'] = 'Obs_' + pond_id + '_' + df_mock['Timestamp'].dt.strftime('%Y%m%d_%H')
    df_mock['Data_Type'] = 'Mock_Data'
    total_len = len(df_mock)
    # Noise nhỏ trên các biến môi trường
    for col in env_cols:
        noise = np.random.normal(loc=0, scale=std[col] * 0.1, size=total_len)
        df_mock[col] = df_mock[col] + noise
    # Baseline
    df_mock['DiseaseOccurrence'] = np.random.choice(
        [1, 2], size=total_len, p=[0.85, 0.15]
    )
    df_mock['OxygenationInterventions'] = 0

    # Outbreak windows (2-3 đợt mỗi pond)
    num_outbreaks = np.random.randint(2, 4)
    for _ in range(num_outbreaks):
        start_idx = np.random.randint(0, total_len - 40)
        duration  = np.random.randint(12, 24)           # 48-96 giờ 
        end_idx   = min(start_idx + duration, total_len - 1)
        burst_len = end_idx - start_idx + 1

        df_mock.loc[start_idx:end_idx, 'DO']   = np.random.uniform(3.0, 5.5, size=burst_len)
        df_mock.loc[start_idx:end_idx, 'Temp'] = np.random.uniform(28.5, 30.5, size=burst_len)
        df_mock.loc[start_idx:end_idx, 'DiseaseOccurrence'] = 2

        # Can thiệp bắt đầu sau 8-16 giờ
        action_start = min(start_idx + np.random.randint(2, 5), end_idx)
        if action_start < end_idx:
            act_len = end_idx - action_start + 1
            df_mock.loc[action_start:end_idx, 'OxygenationInterventions'] = 1

    # Clip đảm bảo không ra giá trị vô lý
    df_mock['pH']        = df_mock['pH'].clip(0, 14)
    df_mock['DO']        = df_mock['DO'].clip(lower=0)
    df_mock['Turbidity'] = df_mock['Turbidity'].clip(lower=0)

    # Lỗi cảm biến (1-3% rows)
    n_shacl_errors = int(total_len * np.random.uniform(0.01, 0.03))
    shacl_idx = np.random.choice(df_mock.index, n_shacl_errors, replace=False)
    half_idx  = len(shacl_idx) // 2
    df_mock.loc[shacl_idx[:half_idx], 'pH'] = np.random.uniform(14.5, 16.0, size=half_idx)
    df_mock.loc[shacl_idx[half_idx:], 'DO'] = -1.5  

    mock_ponds.append(df_mock)

df_all = pd.concat([df_raw, df_pond02, df_pond03] + mock_ponds, ignore_index=True)
df_all = df_all.sort_values(by=['Timestamp', 'Pond_ID']).reset_index(drop=True)

print(f"\nCác Pond trong df_all: {sorted(df_all['Pond_ID'].unique())}")
print(f"Data_Type phân bố:\n{df_all.groupby(['Pond_ID','Data_Type']).size().to_string()}")
print(f"\nTổng rows: {len(df_all)}")
print(f"DiseaseOccurrence unique: {sorted(df_all['DiseaseOccurrence'].unique())}")
print(f"OxygenationInterventions unique: {sorted(df_all['OxygenationInterventions'].unique())}")
display(df_all.head(5))
display(df_pond03.head(5))
display(df_pond02.head(5))


Các Pond trong df_all: ['Pond_01', 'Pond_02', 'Pond_03', 'Pond_04', 'Pond_05']
Data_Type phân bố:
Pond_ID  Data_Type
Pond_01  Real_Data    1096
Pond_02  Real_Data    1080
Pond_03  Real_Data    1087
Pond_04  Mock_Data    1096
Pond_05  Mock_Data    1096

Tổng rows: 5455
DiseaseOccurrence unique: [np.float64(2.0), np.float64(nan), np.float64(0.0), np.float64(1.0)]
OxygenationInterventions unique: [np.float64(0.0), np.float64(1.0), np.float64(nan)]


,Timestamp,Temp,pH,Turbidity,DO,OxygenationInterventions,DiseaseOccurrence,Pond_ID,Data_Type,Observation_ID
0,2025-12-08,27.470000,7.980000,3.300000,6.340000,1.0,2.0,Pond_01,Real_Data,Obs_Pond_01_20251208_00
1,2025-12-08,27.470000,7.980000,3.300000,6.340000,0.0,2.0,Pond_02,Real_Data,Obs_Pond_02_20251208_00
2,2025-12-08,27.679375,7.888962,3.609886,5.950694,NaN,NaN,Pond_03,Real_Data,Obs_Pond_03_20251208_00
3,2025-12-08,27.500788,8.023405,3.272934,6.344966,0.0,1.0,Pond_04,Mock_Data,Obs_Pond_04_20251208_00
4,2025-12-08,27.414603,8.004733,3.326649,6.389588,0.0,1.0,Pond_05,Mock_Data,Obs_Pond_05_20251208_00


,Timestamp,Temp,pH,Turbidity,DO,Pond_ID,Data_Type,Observation_ID
0,2025-12-08 00:00:00,27.679375,7.888962,3.609886,5.950694,Pond_03,Real_Data,Obs_Pond_03_20251208_00
1,2025-12-08 04:00:00,27.581754,7.925377,3.492159,5.678268,Pond_03,Real_Data,Obs_Pond_03_20251208_04
2,2025-12-08 08:00:00,27.171591,7.815301,3.353915,5.896264,Pond_03,Real_Data,Obs_Pond_03_20251208_08
3,2025-12-08 12:00:00,26.706221,7.914152,3.334707,6.705559,Pond_03,Real_Data,Obs_Pond_03_20251208_12
4,2025-12-08 16:00:00,26.847163,7.890615,3.641894,6.405233,Pond_03,Real_Data,Obs_Pond_03_20251208_16


,Timestamp,Temp,pH,Turbidity,DO,OxygenationInterventions,DiseaseOccurrence,Pond_ID,Data_Type,Observation_ID
0,2025-12-08 00:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20251208_00
1,2025-12-08 04:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20251208_04
2,2025-12-08 08:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20251208_08
3,2025-12-08 12:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20251208_12
4,2025-12-08 16:00:00,27.47,7.98,3.3,6.34,0,2,Pond_02,Real_Data,Obs_Pond_02_20251208_16


# 4. Load dữ liệu vào Knowledge Graph và gắn Provenance

Chuyển dữ liệu dạng bảng sang RDF triples theo ontology SOSA và PROV-O.

In [31]:
import pandas as pd
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, XSD, OWL, PROV, DCTERMS
import datetime
import urllib.parse

g = Graph() 
onto_iri = "http://www.ntu.edu.vn/ontology/aqua-risk#"
EX = Namespace(onto_iri)
SOSA = Namespace("http://www.w3.org/ns/sosa/")
g.bind("ex", EX)
g.bind("sosa", SOSA)
g.bind("dcterms", DCTERMS)
g.bind("prov", PROV)
g.bind("owl", OWL)

current_time = datetime.datetime.now().isoformat()

dataset_uri = EX["Dataset_Mendeley_2024"]
g.add((dataset_uri, RDF.type, PROV.Entity))
g.add((dataset_uri, DCTERMS.source, Literal("Fish Health and Water Quality Dataset (Mendeley, 2024)", datatype=XSD.string)))
g.add((dataset_uri, DCTERMS.created, Literal(current_time, datatype=XSD.dateTime)))

simulator_uri = EX["Mock_Data_Generator_V1"]
g.add((simulator_uri, RDF.type, PROV.SoftwareAgent))
g.add((simulator_uri, DCTERMS.description, Literal("Python script generating synthetic ponds based on Pond 1 distribution", datatype=XSD.string)))
g.add((simulator_uri, PROV.wasDerivedFrom, dataset_uri))
g.add((URIRef(onto_iri), PROV.wasDerivedFrom, dataset_uri))

data_props = [
    'hasTemp', 'hasDO', 'hasPH', 'hasTurbidity',
    'hasOxygenationInterventions',
    'hasDiseaseOccurrence', 'observedAtTime',
    'hasDataConfidenceScore', 
    'hasSHACL_ViolationCount',  
]
for p in data_props:
    g.add((EX[p], RDF.type, OWL.DatatypeProperty))

object_props = [
    'hasObservation', 'hasDiseaseEvent', 'hasAction',
    'hasRiskLevel', 'hasRiskFactor', 'hasWaterParam',
    'hasDataQuality',
]
for p in object_props:
    g.add((EX[p], RDF.type, OWL.ObjectProperty))

# Khai báo class SensorDataQuality để gắn chất lượng dữ liệu cảm biến dựa trên SHACL
# Verified: không có vi phạm SHACL (confidence = 1.0)
# Degraded: 1 vi phạm  (confidence = 0.5)
# Unreliable: >=2 vi phạm (confidence = 0.0)
g.add((EX.SensorDataQuality, RDF.type, OWL.Class))
g.add((EX.SensorDataQuality, DCTERMS.description,
       Literal("Represents the trustworthiness of sensor readings based on SHACL validation results.", datatype=XSD.string)))

for qname, score, desc in [
    ("Verified",   1.0, "All SHACL constraints satisfied. Data is fully reliable."),
    ("Degraded",   0.5, "One SHACL violation detected. Data should be used with caution."),
    ("Unreliable", 0.0, "Two or more SHACL violations. Data confidence is low."),
]:
    q_uri = EX[qname]
    g.add((q_uri, RDF.type, EX.SensorDataQuality))
    g.add((q_uri, EX.hasDataConfidenceScore, Literal(score, datatype=XSD.float)))
    g.add((q_uri, DCTERMS.description, Literal(desc, datatype=XSD.string)))

# Chuyển đổi từng row thành RDF triples, tính toán vi phạm SHACL và gắn chất lượng dữ liệu phục vụ ML
def compute_shacl_violations(row):
    violations = 0
    if pd.isna(row['pH']) or not (0 <= row['pH'] <= 14):
        violations += 1
    if pd.isna(row['DO']) or row['DO'] < 0:
        violations += 1
    if pd.isna(row['Temp']):
        violations += 1
    if pd.isna(row['Turbidity']) or row['Turbidity'] < 0:
        violations += 1

    if violations == 0:
        quality_uri = EX["Verified"]
        confidence  = 1.0
    elif violations == 1:
        quality_uri = EX["Degraded"]
        confidence  = 0.5
    else:
        quality_uri = EX["Unreliable"]
        confidence  = 0.0

    return violations, quality_uri, confidence

for index, row in df_all.iterrows():
    pond_id = urllib.parse.quote(str(row['Pond_ID']))
    obs_id  = urllib.parse.quote(str(row['Observation_ID']))
    pond_uri = EX[pond_id]
    g.add((pond_uri, RDF.type, EX.Pond))
    obs_uri = EX[obs_id]
    g.add((obs_uri, RDF.type, SOSA.Observation))
    g.add((pond_uri, EX.hasObservation, obs_uri))
    timestamp_literal = Literal(row['Timestamp'].isoformat(), datatype=XSD.dateTime)
    g.add((obs_uri, EX.observedAtTime, timestamp_literal))
    # Thông số vật lý
    g.add((obs_uri, EX.hasTemp,      Literal(row['Temp'],      datatype=XSD.float)))
    g.add((obs_uri, EX.hasDO,        Literal(row['DO'],        datatype=XSD.float)))
    g.add((obs_uri, EX.hasPH,        Literal(row['pH'],        datatype=XSD.float)))
    g.add((obs_uri, EX.hasTurbidity, Literal(row['Turbidity'], datatype=XSD.float)))

    # Sự kiện bệnh
    disease_val = row.get('DiseaseOccurrence')
    if pd.notna(disease_val) and disease_val > 0:
        disease_uri = EX[f"DiseaseEvent_{obs_id}"]
        g.add((disease_uri, RDF.type, EX.DiseaseEvent))
        g.add((disease_uri, EX.hasDiseaseOccurrence, Literal(int(disease_val), datatype=XSD.integer)))
        g.add((obs_uri, EX.hasDiseaseEvent, disease_uri))

    # Sự kiện can thiệp
    oxy_val = row.get('OxygenationInterventions')
    if pd.notna(oxy_val):
        g.add((obs_uri, EX.hasOxygenationInterventions, Literal(int(oxy_val), datatype=XSD.integer)))
        if oxy_val > 0:
            action_oxy_uri = EX[f"Action_Oxy_{obs_id}"]
            g.add((action_oxy_uri, RDF.type, EX.Oxygenation))
            g.add((pond_uri, EX.hasAction, action_oxy_uri))

    n_violations, quality_uri, confidence = compute_shacl_violations(row)
    g.add((obs_uri, EX.hasDataQuality, quality_uri))
    g.add((obs_uri, EX.hasSHACL_ViolationCount,
           Literal(n_violations, datatype=XSD.integer)))
    g.add((obs_uri, EX.hasDataConfidenceScore,
           Literal(confidence, datatype=XSD.float)))

    if row['Data_Type'] == 'Real_Data':
        g.add((obs_uri, PROV.wasDerivedFrom, dataset_uri))
    else:
        g.add((obs_uri, PROV.wasGeneratedBy, simulator_uri))
    g.add((obs_uri, PROV.generatedAtTime, Literal(current_time, datatype=XSD.dateTime)))

output_file = "aqua_kg_multiPond.ttl"
g.serialize(destination=output_file, format='turtle')
print(f"Tổng triples: {len(g)}")

from collections import Counter
class_counter = Counter()
for s, p, o in g.triples((None, RDF.type, None)):
    class_counter[str(o).split("#")[-1]] += 1

print("\nSố instances theo từng class:")
for cls, count in sorted(class_counter.items()):
    print(f"  {cls}: {count}")


Tổng triples: 84161

Số instances theo từng class:
  Class: 1
  DatatypeProperty: 9
  DiseaseEvent: 4296
  Entity: 1
  ObjectProperty: 7
  Oxygenation: 703
  Pond: 5
  SensorDataQuality: 3
  SoftwareAgent: 1
  http://www.w3.org/ns/sosa/Observation: 5455


# 4.1. Kiểm tra ràng buộc dữ liệu (SHACL Validation)

Việc kiểm tra dữ liệu rác bằng SHACL được thực thi độc lập qua thư viện `pyshacl` tại file `shacl.py`. Đầu vào là file `aqua_kg_multiPond.ttl`. Kết quả các dòng vi phạm sẽ được đối chiếu để sinh ra cờ cảnh báo lỗi cảm biến ở các bước sau.

# 4.2. Suy luận tri thức sinh thái (Pellet Reasoner)

File đồ thị gốc `aqua_kg_multiPond.ttl` được nạp vào phần mềm Protégé để thực thi bộ luật SWRL thông qua Pellet Reasoner. Hệ thống tự động gán nhãn rủi ro (RiskLevel, RiskFactor). Kết quả đồ thị đã suy luận được xuất ra thành file `aqua-inferred.ttl` để chuẩn bị cho bước trích xuất đặc trưng.

# 5. Truy vấn SPARQL và trích xuất ontology-enhanced features

Đọc ontology sau reasoning và trích xuất các đặc trưng ngữ nghĩa phục vụ ML.

In [32]:
import pandas as pd
import numpy as np
from rdflib import Graph

inferred_file = "aqua_inferred.ttl"
g_inferred = Graph()
g_inferred.parse(inferred_file, format="turtle")

query = """
PREFIX ex:   <http://www.ntu.edu.vn/ontology/aqua-risk#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

SELECT ?obs_id ?risk_level (COUNT(?rf) AS ?risk_factors_count) ?confidence_score ?shacl_violations
WHERE {
    ?obs rdf:type sosa:Observation .
    BIND(STRAFTER(STR(?obs), "#") AS ?obs_id)
    OPTIONAL {
        ?obs ex:hasRiskLevel ?rl_uri . BIND(STRAFTER(STR(?rl_uri), "#") AS ?risk_level)
    }
    OPTIONAL { ?obs ex:hasRiskFactor ?rf }
    OPTIONAL { ?obs ex:hasDataConfidenceScore ?confidence_score }
    OPTIONAL { ?obs ex:hasSHACL_ViolationCount ?shacl_violations }
}
GROUP BY ?obs_id ?risk_level ?confidence_score ?shacl_violations
"""

qres = g_inferred.query(query)
kg_data = []
for row in qres:
    kg_data.append({
        'Observation_ID':      str(row.obs_id),
        'RiskLevel':           str(row.risk_level)       if row.risk_level       else None,
        'RiskFactorsCount':    int(row.risk_factors_count) if row.risk_factors_count else 0,
        'DataConfidenceScore': float(row.confidence_score) if row.confidence_score else None,
        'SHACL_ViolationCount_KG': int(row.shacl_violations) if row.shacl_violations else 0,
    })

df_kg_features = pd.DataFrame(kg_data)

df_all = pd.merge(df_all, df_kg_features, on='Observation_ID', how='left')

# Nếu SWRL không suy diễn được (ví dụ: sensor Unreliable làm dữ
# liệu không khớp rule), gán "Unknown" để phân biệt rõ với Low thực.
df_all['RiskLevel'] = df_all['RiskLevel'].fillna('Unknown')

risk_level_map = {'Unknown': -1, 'Low': 0, 'Medium': 1, 'High': 2}
df_all['RiskLevel_encoded'] = df_all['RiskLevel'].map(risk_level_map).fillna(-1).astype(int)

# ── SHACL violations: tính lại từ giá trị cảm biến
df_all['SHACL_pH_Violation']  = ((df_all['pH'] > 14) | (df_all['pH'] < 0)).astype(int)
df_all['SHACL_DO_Violation']  = (df_all['DO'] < 0).astype(int)
df_all['SHACL_Temp_Missing']  = (df_all['Temp'].isna()).astype(int)
df_all['SHACL_Turb_Violation'] = (df_all['Turbidity'] < 0).astype(int)

df_all['SHACL_Violation_Count'] = (
    df_all['SHACL_pH_Violation'] +
    df_all['SHACL_DO_Violation'] +
    df_all['SHACL_Temp_Missing'] +
    df_all['SHACL_Turb_Violation']
)
df_all['SHACL_Sensor_Error_Flag'] = (df_all['SHACL_Violation_Count'] > 0).astype(int)

# DataonfidenceScore: từ KG SensorDataQuality 
# Giá trị [0.0–1.0] phản ánh mức tin cậy cảm biến được suy diễn
# trong ontology dùng làm feature cho ML và điều chỉnh ngữ nghĩa RiskLevel.
df_all['DataConfidenceScore'] = df_all['DataConfidenceScore'].fillna(1.0) 

# Flags rủi ro từ KG 
df_all['HighRisk_Triggered']    = (df_all['RiskLevel'] == 'High').astype(int)
df_all['MediumRisk_Triggered']  = (df_all['RiskLevel'] == 'Medium').astype(int)
df_all['RiskFactorsCount']      = df_all['RiskFactorsCount'].fillna(0).astype(int)

output_file = "Dataset_Mocked_MultiPond.csv"
df_all.to_csv(output_file, index=False)
print(f"\nĐã lưu: {output_file}")

print("Total triples:", len(g))



Đã lưu: Dataset_Mocked_MultiPond.csv
Total triples: 84161


# 6. Tạo Sliding Window cho dữ liệu chuỗi thời gian

Tạo chuỗi đặc trưng T-1 → T-6 cho bài toán dự báo.

In [33]:
import numpy as np

df_all['y_classification'] = (df_all['DiseaseOccurrence'] >= P_pandemic_disease_threshold).astype(int)
df_all['y_regression']     = df_all['DiseaseOccurrence']

env_cols   = ['Temp', 'DO', 'pH', 'Turbidity']
interv_cols = ['OxygenationInterventions']
kg_cols = [
    'RiskLevel_encoded',       
    'DataConfidenceScore',    
    'RiskFactorsCount',
    'HighRisk_Triggered',
    'MediumRisk_Triggered',   
    'SHACL_Violation_Count',
    'SHACL_pH_Violation',
    'SHACL_DO_Violation',
    'SHACL_Temp_Missing',
    'SHACL_Turb_Violation',
]

cfg1_features = []
cfg2_features = []
cfg3_features = []
cfg4_features = []

window_steps = [1, 2, 3, 4, 5, 6]

# CFG1: chỉ môi trường
for col in env_cols:
    for i in window_steps:
        t_col = f"{col}_t-{i}"
        cfg1_features.append(t_col)
        df_all[t_col] = df_all.groupby('Pond_ID')[col].shift(i)

# CFG2: môi trường + can thiệp
cfg2_features = cfg1_features.copy()
for col in interv_cols:
    for i in window_steps:
        t_col = f"{col}_t-{i}"
        cfg2_features.append(t_col)
        df_all[t_col] = df_all.groupby('Pond_ID')[col].shift(i)

# CFG3: toàn diện
cfg3_features = cfg2_features.copy()
for col in kg_cols:
    if col in df_all.columns:
        for i in window_steps:
            t_col = f"{col}_t-{i}"
            cfg3_features.append(t_col)
            df_all[t_col] = df_all.groupby('Pond_ID')[col].shift(i)

# CFG4: môi trường + KG 
cfg4_features = cfg1_features.copy()
for col in kg_cols:
    if col in df_all.columns:
        for i in window_steps:
            t_col = f"{col}_t-{i}"
            if t_col not in cfg4_features:
                cfg4_features.append(t_col)
            df_all[t_col] = df_all.groupby('Pond_ID')[col].shift(i)

cfg4_noDCS_features = [f for f in cfg4_features if 'DataConfidenceScore' not in f]

# Loại bỏ NaN từ lag features
df_train = df_all[(df_all['Pond_ID'] != 'Pond_03') & (df_all['Data_Type'] == 'Real_Data')].copy()
df_train = df_train.dropna(subset=cfg3_features).reset_index(drop=True)

# Kiểm thử tính linh hoạt: Lấy Dataset 3 (khuyết can thiệp/bệnh)
df_d3 = df_all[df_all['Pond_ID'] == 'Pond_03'].copy()
df_d3 = df_d3.dropna(subset=cfg4_features).reset_index(drop=True)

df_dashboard = df_all.copy()

print(f"→ Tập huấn luyện chính (Pond_01)      : {len(df_train)} dòng")
print(f"→ Tập kiểm thử linh hoạt (Pond_03 - D3): {len(df_d3)} dòng")
print(f"→ Tập Dashboard (All Ponds)            : {len(df_dashboard)} dòng")




→ Tập huấn luyện chính (Pond_01)      : 2506 dòng
→ Tập kiểm thử linh hoạt (Pond_03 - D3): 1082 dòng
→ Tập Dashboard (All Ponds)            : 6355 dòng


# 7. Chia tập train/test

Tách dữ liệu huấn luyện và kiểm thử cho Machine Learning.

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from sklearn.base import clone
warnings.filterwarnings('ignore')

y_clf_real = df_train['y_classification']
y_reg_real = df_train['y_regression']

X_train_cfg1_c, X_test_cfg1_c, y_train_clf, y_test_clf = train_test_split(
    df_train[cfg1_features], y_clf_real, test_size=0.2, shuffle=False)
X_train_cfg2_c, X_test_cfg2_c, _, _ = train_test_split(df_train[cfg2_features], y_clf_real, test_size=0.2, shuffle=False)
X_train_cfg3_c, X_test_cfg3_c, _, _ = train_test_split(df_train[cfg3_features], y_clf_real, test_size=0.2, shuffle=False)
X_train_cfg4_c, X_test_cfg4_c, _, _ = train_test_split(df_train[cfg4_features], y_clf_real, test_size=0.2, shuffle=False)

X_train_noDCS_r, X_test_noDCS_r, _, _ = train_test_split(
    df_train[cfg4_noDCS_features], y_reg_real, test_size=0.2, shuffle=False)

configs_clf = {
    "CFG1 (Chỉ Môi trường)": (X_train_cfg1_c, X_test_cfg1_c),
    "CFG2 (MT + Can thiệp)": (X_train_cfg2_c, X_test_cfg2_c),
    "CFG4 (MT + Ontology)": (X_train_cfg4_c, X_test_cfg4_c),
    "CFG3 (Toàn diện)": (X_train_cfg3_c, X_test_cfg3_c)
}

X_train_cfg1_r, X_test_cfg1_r, y_train_reg, y_test_reg = train_test_split(df_train[cfg1_features], y_reg_real, test_size=0.2, shuffle=False)
X_train_cfg2_r, X_test_cfg2_r, _, _ = train_test_split(df_train[cfg2_features], y_reg_real, test_size=0.2, shuffle=False)
X_train_cfg3_r, X_test_cfg3_r, _, _ = train_test_split(df_train[cfg3_features], y_reg_real, test_size=0.2, shuffle=False)
X_train_cfg4_r, X_test_cfg4_r, _, _ = train_test_split(df_train[cfg4_features], y_reg_real, test_size=0.2, shuffle=False)

configs_reg = {
    "CFG1 (Chỉ Môi trường)": (X_train_cfg1_r, X_test_cfg1_r),
    "CFG2 (MT + Can thiệp)": (X_train_cfg2_r, X_test_cfg2_r),
    "CFG4 (MT + Ontology)": (X_train_cfg4_r, X_test_cfg4_r),
    "CFG3 (Toàn diện)": (X_train_cfg3_r, X_test_cfg3_r),
    "CFG4-noDCS": (X_train_noDCS_r, X_test_noDCS_r)
}

# 8. Huấn luyện và đánh giá mô hình Classification

Huấn luyện các mô hình phân loại cảnh báo sớm dịch bệnh.

In [35]:
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

for model_name, model in clf_models.items():
    print(f"\n▶ TÊN MÔ HÌNH: {model_name}")
    print("-" * 95)
    for cfg_name, (X_tr, X_te) in configs_clf.items():
        current_model = clone(model)
        current_model.fit(X_tr, y_train_clf)
        y_pred = current_model.predict(X_te)
        
        try:
            y_prob = current_model.predict_proba(X_te)[:, 1]
            roc = roc_auc_score(y_test_clf, y_prob)
        except:
            roc = 0.0 
            
        acc = accuracy_score(y_test_clf, y_pred)
        prec = precision_score(y_test_clf, y_pred, zero_division=0)
        rec = recall_score(y_test_clf, y_pred, zero_division=0)
        f1 = f1_score(y_test_clf, y_pred, zero_division=0)
        
        print(f"{cfg_name: <30} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f}")


▶ TÊN MÔ HÌNH: Logistic Regression
-----------------------------------------------------------------------------------------------
CFG1 (Chỉ Môi trường)          | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan
CFG2 (MT + Can thiệp)          | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan
CFG4 (MT + Ontology)           | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan
CFG3 (Toàn diện)               | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan

▶ TÊN MÔ HÌNH: Random Forest
-----------------------------------------------------------------------------------------------
CFG1 (Chỉ Môi trường)          | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan
CFG2 (MT + Can thiệp)          | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan
CFG4 (MT + Ontology)           | Acc: 1.0000 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | ROC-AUC: nan
CFG3 (Toàn diện)            

# 9. Huấn luyện và đánh giá mô hình Regression

Huấn luyện các mô hình hồi quy dự báo số ca bệnh.

In [36]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor": XGBRegressor(random_state=42)
}

for model_name, model in reg_models.items():
    print(f"\n▶ TÊN MÔ HÌNH: {model_name}")
    print("-" * 85)
    for cfg_name, (X_tr, X_te) in configs_reg.items():
        current_model = clone(model)
        current_model.fit(X_tr, y_train_reg)
        y_pred = current_model.predict(X_te)
        
        mae = mean_absolute_error(y_test_reg, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred)) 
        r2 = r2_score(y_test_reg, y_pred)
        
        print(f"{cfg_name: <30} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")


▶ TÊN MÔ HÌNH: Linear Regression
-------------------------------------------------------------------------------------
CFG1 (Chỉ Môi trường)          | MAE: 1.1878 | RMSE: 1.3527 | R²: -51.9324
CFG2 (MT + Can thiệp)          | MAE: 1.1914 | RMSE: 1.3527 | R²: -51.9317
CFG4 (MT + Ontology)           | MAE: 1.2468 | RMSE: 1.4344 | R²: -58.5161
CFG3 (Toàn diện)               | MAE: 1.1806 | RMSE: 1.3513 | R²: -51.8213
CFG4-noDCS                     | MAE: 1.2468 | RMSE: 1.4344 | R²: -58.5161

▶ TÊN MÔ HÌNH: Random Forest Regressor
-------------------------------------------------------------------------------------
CFG1 (Chỉ Môi trường)          | MAE: 0.0359 | RMSE: 0.1894 | R²: -0.0372
CFG2 (MT + Can thiệp)          | MAE: 0.0361 | RMSE: 0.1894 | R²: -0.0372
CFG4 (MT + Ontology)           | MAE: 0.0163 | RMSE: 0.0940 | R²: 0.7446
CFG3 (Toàn diện)               | MAE: 0.0161 | RMSE: 0.0927 | R²: 0.7513
CFG4-noDCS                     | MAE: 0.0164 | RMSE: 0.0936 | R²: 0.7465

▶ TÊN MÔ HÌ

## 10. Đánh giá tính linh hoạt của hệ thống trên tập dữ liệu khuyết (Pond_03)

In [37]:
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import numpy as np

print("▶ SỬ DỤNG MÔ HÌNH XGBOOST ĐỂ PREDICT TRÊN D3")
print("-" * 70)

xgb_model_cfg1 = XGBRegressor(random_state=42)
xgb_model_cfg1.fit(df_train[cfg1_features], df_train['y_regression'])

xgb_model_cfg4 = XGBRegressor(random_state=42)
xgb_model_cfg4.fit(df_train[cfg4_features], df_train['y_regression'])

X_d3_cfg1 = df_d3[cfg1_features]
X_d3_cfg4 = df_d3[cfg4_features]

# D3 là ao chỉ có thông số môi trường, không ghi nhận bệnh -> nhãn thực tế = 0
y_true_d3 = np.zeros(len(df_d3))

pred_d3_cfg1 = xgb_model_cfg1.predict(X_d3_cfg1)
pred_d3_cfg4 = xgb_model_cfg4.predict(X_d3_cfg4)

mae_cfg1 = mean_absolute_error(y_true_d3, pred_d3_cfg1)
mae_cfg4 = mean_absolute_error(y_true_d3, pred_d3_cfg4)
rmae_cfg1 = np.sqrt(mean_squared_error(y_true_d3, pred_d3_cfg1))
rmae_cfg4 = np.sqrt(mean_squared_error(y_true_d3, pred_d3_cfg4))

print(f"Chỉ Môi trường: MAE = {mae_cfg1:.4f}, RMAE = {rmae_cfg1:.4f}")
print(f"MT + Ontology: MAE = {mae_cfg4:.4f}, RMAE = {rmae_cfg4:.4f}")

▶ SỬ DỤNG MÔ HÌNH XGBOOST ĐỂ PREDICT TRÊN D3
----------------------------------------------------------------------
Chỉ Môi trường: MAE = 1.0544, RMAE = 1.0757
MT + Ontology: MAE = 0.8932, RMAE = 1.0019


In [38]:
import joblib
from sklearn.ensemble import RandomForestClassifier

best_clf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

best_clf_model.fit(df_train[cfg4_features], y_clf_real)

joblib.dump(best_clf_model, "best_classifier_model.pkl")

joblib.dump(cfg4_features, "classifier_features.pkl")

print("Đã lưu Classification Model")


Đã lưu Classification Model


In [39]:
import joblib
from xgboost import XGBRegressor

best_reg_model = XGBRegressor(
    random_state=42
)

best_reg_model.fit(df_train[cfg4_features], y_reg_real)

joblib.dump(best_reg_model, "best_regressor_model.pkl")

joblib.dump(cfg4_features, "regressor_features.pkl")

print("Đã lưu Regression Model")


Đã lưu Regression Model
